# Retrieval Test

This notebook corresponds to `src/vector_store/retrieve.py`.

Purpose:

```text
query  →  query embedding  →  Chroma similarity search  →  top K chunks
```

This notebook does not call Claude. It only tests whether semantic retrieval can find relevant chunks from the Chroma vector database.

In [ ]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHROMA_DIR = PROJECT_ROOT / "Data" / "chroma_db"

print("Chroma directory exists:", CHROMA_DIR.exists())

In [ ]:
class ChromaRetriever:
    """
    Retrieve top-k relevant chunks from Chroma using semantic similarity.
    """

    def __init__(
        self,
        chroma_dir: Path,
        collection_name: str = "rag_documents",
        model_name: str = "all-MiniLM-L6-v2",
    ):
        self.chroma_dir = chroma_dir
        self.collection_name = collection_name
        self.model_name = model_name

        self.model = SentenceTransformer(model_name)

        self.client = chromadb.PersistentClient(path=str(chroma_dir))
        self.collection = self.client.get_collection(name=collection_name)

    def retrieve(self, query: str, top_k: int = 5) -> list:
        """
        Retrieve top-k chunks for a query.
        """
        query_embedding = self.model.encode([query]).tolist()[0]

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
        )

        retrieved = []

        for i in range(len(results["ids"][0])):
            item = {
                "rank": i + 1,
                "chunk_id": results["ids"][0][i],
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "distance": results["distances"][0][i],
            }

            retrieved.append(item)

        return retrieved


def get_default_retriever() -> ChromaRetriever:
    """
    Create a retriever using default project paths.
    """
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    chroma_dir = project_root / "Data" / "chroma_db"

    return ChromaRetriever(
        chroma_dir=chroma_dir,
        collection_name="rag_documents",
        model_name="all-MiniLM-L6-v2",
    )


def retrieve(query: str, top_k: int = 5) -> list:
    """
    Convenience function used by later RAG pipeline.
    """
    retriever = get_default_retriever()
    return retriever.retrieve(query, top_k=top_k)

In [ ]:
retriever = get_default_retriever()
print("Retriever is ready.")

In [ ]:
test_query = "What is net present value?"
results = retriever.retrieve(test_query, top_k=5)

print(f"Query: {test_query}")
print("=" * 80)

for item in results:
    print(f"Rank: {item['rank']}")
    print(f"Chunk ID: {item['chunk_id']}")
    print(f"Title: {item['metadata'].get('title')}")
    print(f"Source: {item['metadata'].get('source_html')}")
    print(f"Distance: {item['distance']}")
    print(item["text"][:800])
    print("=" * 80)

In [ ]:
retrieval_rows = []
for item in results:
    retrieval_rows.append({
        "rank": item["rank"],
        "chunk_id": item["chunk_id"],
        "doc_id": item["metadata"].get("doc_id"),
        "title": item["metadata"].get("title"),
        "distance": item["distance"],
        "text_preview": item["text"][:250],
    })

pd.DataFrame(retrieval_rows)

In [ ]:
test_queries = [
    "What is net present value?",
    "What is portfolio diversification?",
    "What is the capital asset pricing model?",
    "What role do central banks play in monetary policy?",
    "What is the time value of money?",
]

for query in test_queries:
    print("\n" + "=" * 100)
    print("Query:", query)
    results = retriever.retrieve(query, top_k=3)
    for item in results:
        print(f"Rank {item['rank']} | {item['metadata'].get('title')} | distance={item['distance']:.4f}")
        print(item["text"][:400].replace("\n", " "))
        print("-" * 100)